In [5]:
import warnings
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error

# --- Global Constants ---
PREDS_PATH = './model_predictions/'
DATA_PATH = './'
OLD_BEST_SCORE = 296350.26 # The score from your winning notebook
N_OPTUNA_TRIALS = 30
COMPETITION_ALPHA = 0.1
RANDOM_STATE = 42

# --- Winkler Score Helper Function (Upgraded) ---
def winkler_score(y_true, lower, upper, alpha=COMPETITION_ALPHA, return_coverage=False):
    """
    Upgraded function to also return coverage and width for diagnostics.
    """
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (lower - y_true) * (2 / alpha), 0)
    penalty_upper = np.where(y_true > upper, (y_true - upper) * (2 / alpha), 0)
    score = np.mean(width + penalty_lower + penalty_upper)
    
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return score, coverage, np.mean(width)
    return score

warnings.filterwarnings('ignore')
print("Libraries and helper functions loaded successfully.")

Libraries and helper functions loaded successfully.


In [6]:
print("--- Loading all pre-trained base model predictions ---")

try:
    # --- Load Mean Model Predictions ---
    oof_xgb_preds = np.load(f'{PREDS_PATH}oof_xgb_preds.npy')
    test_xgb_preds = np.load(f'{PREDS_PATH}test_xgb_preds.npy')
    oof_cb_preds = np.load(f'{PREDS_PATH}oof_cb_preds.npy')
    test_cb_preds = np.load(f'{PREDS_PATH}test_cb_preds.npy')
    oof_nn_preds = np.load(f'{PREDS_PATH}oof_nn_preds.npy')
    test_nn_preds = np.load(f'{PREDS_PATH}test_nn_preds.npy')
    print("All MEAN model predictions loaded.")
    
    # --- Load Ground Truth Target ---
    y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
    print("Ground truth target 'y_true' loaded.")

except FileNotFoundError as e:
    print(f"\nERROR: Could not find a prediction file. {e}")
    print("Please ensure you have run all training notebooks and saved their predictions first.")

print("\nAll predictions loaded successfully. Ready to build the final ensemble.")

--- Loading all pre-trained base model predictions ---
All MEAN model predictions loaded.
Ground truth target 'y_true' loaded.

All predictions loaded successfully. Ready to build the final ensemble.


In [7]:
print("\n--- STAGE 1: Finding Optimal Weights for the 3-Model MEAN Ensemble ---")

# Stack the OOF predictions into a single array for the optimizer
oof_preds_stack_mean = np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T

# Define the objective function (RMSE) for the mean ensemble
def get_ensemble_rmse(weights):
    final_prediction = np.dot(oof_preds_stack_mean, weights)
    return np.sqrt(mean_squared_error(y_true, final_prediction))

# Run the optimizer to find the best blending weights
result_mean = minimize(get_ensemble_rmse, [1/3]*3, method='SLSQP',
                       bounds=[(0,1)]*3,
                       constraints=({'type': 'eq', 'fun': lambda w: 1 - sum(w)}))

best_mean_weights = result_mean.x
best_ensemble_rmse = result_mean.fun

# Create the final blended mean predictions using the optimal weights
oof_ensemble_mean = np.dot(oof_preds_stack_mean, best_mean_weights)
test_ensemble_mean = np.dot(np.vstack([test_xgb_preds, test_cb_preds, test_nn_preds]).T, best_mean_weights)

print("\n" + "="*50)
print("      OPTIMAL 3-MODEL MEAN BLEND SEARCH COMPLETE")
print("="*50)
print(f"Lowest Ensemble RMSE achieved: ${best_ensemble_rmse:,.2f}")
print(f"Optimal Weight for XGBoost : {best_mean_weights[0]:.4f}")
print(f"Optimal Weight for CatBoost: {best_mean_weights[1]:.4f}")
print(f"Optimal Weight for NeuralNet: {best_mean_weights[2]:.4f}")
print("="*50)
print("\nSuper-ensemble mean predictions are now created and ready for use as meta-features.")


--- STAGE 1: Finding Optimal Weights for the 3-Model MEAN Ensemble ---

      OPTIMAL 3-MODEL MEAN BLEND SEARCH COMPLETE
Lowest Ensemble RMSE achieved: $96,778.10
Optimal Weight for XGBoost : 0.4100
Optimal Weight for CatBoost: 0.4779
Optimal Weight for NeuralNet: 0.1121

Super-ensemble mean predictions are now created and ready for use as meta-features.


In [11]:
print("\n--- STAGE 2: Tuning the ENHANCED Quantile Stacking Meta-Model ---")

# --- Load Original Features for Context ---
# We need to give the meta-model more context about the properties.
# We will use a lightweight version of the feature engineering from your other notebooks.
print("Loading original raw data for feature engineering...")
df_train_raw = pd.read_csv(DATA_PATH + 'dataset.csv')
# Select only the most powerful, non-leaky features
context_features = [
    'grade', 'sqft', 'lat', 'long', 'yr_built', 'yr_renovated' 
]
# For simplicity, we will assume these column names. Adjust if yours are different.
# Simple renaming for consistency if needed:
df_train_raw = df_train_raw.rename(columns={'latitude': 'lat', 'longitude': 'long', 'year_built': 'yr_built', 'year_reno':'yr_renovated'})
X_context = df_train_raw[['grade', 'sqft', 'lat', 'long', 'yr_built', 'yr_renovated']].fillna(0)
X_context['property_age'] = 2025 - X_context['yr_built']


# --- Create the ENHANCED Feature Set for the Meta-Model ---
# We combine the base model predictions with the powerful context features.
X_meta_train_enhanced = pd.concat([
    pd.DataFrame({
        'xgb_pred': oof_xgb_preds,
        'cb_pred': oof_cb_preds,
        'nn_pred': oof_nn_preds,
        'ensemble_mean': oof_ensemble_mean
    }),
    X_context.reset_index(drop=True) # Ensure indices align
], axis=1)


print(f"Enhanced meta-model training data created with shape: {X_meta_train_enhanced.shape}")

# --- Create a validation set for tuning ---
from sklearn.model_selection import train_test_split
X_meta_train_tune, X_meta_val_tune, y_meta_train_tune, y_meta_val_tune = train_test_split(
    X_meta_train_enhanced, y_true, test_size=0.25, random_state=RANDOM_STATE
)

# --- Define the Optuna Objective Function ---
def objective_winkler_meta_enhanced(trial):
    params = {
        'objective': 'reg:quantileerror', 'tree_method': 'hist', 'n_jobs': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'lambda': trial.suggest_float('lambda', 0.1, 20.0, log=True),
        'alpha': trial.suggest_float('alpha', 0.1, 20.0, log=True),
        'n_estimators': 1000
    }
    # ... (rest of the objective function is the same as before) ...
    params['quantile_alpha'] = 0.05
    model_lower = xgb.XGBRegressor(**params, early_stopping_rounds=30)
    model_lower.fit(X_meta_train_tune, y_meta_train_tune, eval_set=[(X_meta_val_tune, y_meta_val_tune)], verbose=False)

    params['quantile_alpha'] = 0.95
    model_upper = xgb.XGBRegressor(**params, early_stopping_rounds=30)
    model_upper.fit(X_meta_train_tune, y_meta_train_tune, eval_set=[(X_meta_val_tune, y_meta_val_tune)], verbose=False)

    lower_preds = model_lower.predict(X_meta_val_tune)
    upper_preds = model_upper.predict(X_meta_val_tune)
    score = winkler_score(y_meta_val_tune, lower_preds, upper_preds)
    return score

# --- Run the Optuna Study ---
print(f"Running Optuna study for {N_OPTUNA_TRIALS} trials on ENHANCED features...")
study_meta = optuna.create_study(direction='minimize')
study_meta.optimize(objective_winkler_meta_enhanced, n_trials=N_OPTUNA_TRIALS)

best_params_meta = study_meta.best_params
print("\n--- Tuning Complete ---")
print(f"Best Winkler Score found during tuning: {study_meta.best_value:,.2f}")
print("Best parameters for meta-model:")
print(best_params_meta)

### **Cell 5: Train Final Meta-Model, Evaluate, and Create Submission (BUG FIXED)**

print("\n--- STAGE 3: Training Final Meta-Model & Final Showdown (BUG FIXED) ---")

# --- Create ENHANCED Meta-Features for the Test Set ---
# We must perform the same feature engineering for the test set.
df_test_raw = pd.read_csv(DATA_PATH + 'test.csv')
df_test_raw = df_test_raw.rename(columns={'latitude': 'lat', 'longitude': 'long', 'year_built': 'yr_built', 'year_reno':'yr_renovated'})
X_context_test = df_test_raw[['grade', 'sqft', 'lat', 'long', 'yr_built', 'yr_renovated']].fillna(0)
X_context_test['property_age'] = 2025 - X_context_test['yr_built']

X_meta_test_enhanced = pd.concat([
    pd.DataFrame({
        'xgb_pred': test_xgb_preds,
        'cb_pred': test_cb_preds,
        'nn_pred': test_nn_preds,
        'ensemble_mean': test_ensemble_mean
    }),
    X_context_test.reset_index(drop=True)
], axis=1)


# --- Train Final Models on ALL OOF Data ---
final_params = best_params_meta.copy()
# ***** THE BUG FIX IS HERE *****
final_params['objective'] = 'reg:quantileerror' 

# Train LOWER bound model (q=0.05)
print("\nTraining FINAL lower bound meta-model (q=0.05) on all OOF data...")
final_params['quantile_alpha'] = 0.05
model_meta_final_lower = xgb.XGBRegressor(**final_params, n_estimators=1200)
model_meta_final_lower.fit(X_meta_train_enhanced, y_true)

# Train UPPER bound model (q=0.95)
print("Training FINAL upper bound meta-model (q=0.95) on all OOF data...")
final_params['quantile_alpha'] = 0.95
model_meta_final_upper = xgb.XGBRegressor(**final_params, n_estimators=1200)
model_meta_final_upper.fit(X_meta_train_enhanced, y_true)

# --- Evaluate the New Stacked Pipeline ---
final_oof_lower = model_meta_final_lower.predict(X_meta_train_enhanced)
final_oof_upper = model_meta_final_upper.predict(X_meta_train_enhanced)
best_score_final, final_coverage, final_width = winkler_score(y_true, final_oof_lower, final_oof_upper, return_coverage=True)

# --- THE ULTIMATE PIPELINE: FINAL SHOWDOWN ---
print("\n" + "="*60)
print("     THE ENHANCED META-MODEL PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Previous Best Score (Error Model) : ${OLD_BEST_SCORE:,.2f}")
print(f"New STACKED PIPELINE Score        : ${best_score_final:,.2f}")
print(f"  - Final OOF Coverage            : {final_coverage:.2%}")
print(f"  - Final OOF Average Width       : ${final_width:,.2f}")
print("="*60)

if best_score_final < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The new context-aware pipeline is the champion!")
else:
    print("\nCONCLUSION: So close! The original pipeline remains superior.")


--- STAGE 2: Tuning the ENHANCED Quantile Stacking Meta-Model ---
Loading original raw data for feature engineering...


[I 2025-07-17 15:04:56,072] A new study created in memory with name: no-name-14863277-354f-4e9e-903f-f7942025f90c


Enhanced meta-model training data created with shape: (200000, 11)
Running Optuna study for 30 trials on ENHANCED features...


[I 2025-07-17 15:05:16,860] Trial 0 finished with value: 318696.74437609373 and parameters: {'learning_rate': 0.012367267586425468, 'max_depth': 4, 'subsample': 0.6340365722109829, 'colsample_bytree': 0.6647968663534393, 'lambda': 3.483681939012966, 'alpha': 1.4006552782715234}. Best is trial 0 with value: 318696.74437609373.
[I 2025-07-17 15:05:33,514] Trial 1 finished with value: 318739.0724546875 and parameters: {'learning_rate': 0.013048627064262602, 'max_depth': 8, 'subsample': 0.7829520804744925, 'colsample_bytree': 0.7538579240313183, 'lambda': 4.330010664165217, 'alpha': 1.029896702856613}. Best is trial 0 with value: 318696.74437609373.
[I 2025-07-17 15:05:44,648] Trial 2 finished with value: 317849.110859375 and parameters: {'learning_rate': 0.05293188536037233, 'max_depth': 6, 'subsample': 0.9312645031150771, 'colsample_bytree': 0.6483663904451159, 'lambda': 0.13828589062604796, 'alpha': 0.3182683233858273}. Best is trial 2 with value: 317849.110859375.
[I 2025-07-17 15:06:0


--- Tuning Complete ---
Best Winkler Score found during tuning: 314,203.21
Best parameters for meta-model:
{'learning_rate': 0.0199899023649032, 'max_depth': 6, 'subsample': 0.823103448049299, 'colsample_bytree': 0.8159084313243358, 'lambda': 6.869908654316057, 'alpha': 10.930230930896247}

--- STAGE 3: Training Final Meta-Model & Final Showdown (BUG FIXED) ---

Training FINAL lower bound meta-model (q=0.05) on all OOF data...
Training FINAL upper bound meta-model (q=0.95) on all OOF data...

     THE ENHANCED META-MODEL PIPELINE: FINAL SHOWDOWN
Previous Best Score (Error Model) : $296,350.26
New STACKED PIPELINE Score        : $290,912.17
  - Final OOF Coverage            : 90.11%
  - Final OOF Average Width       : $213,878.75

CONCLUSION: VICTORY! The new context-aware pipeline is the champion!


In [14]:
import joblib
import os

# --- Define a path to save the models ---
MODELS_SAVE_PATH = './final_models/'

# Create the directory if it doesn't exist
os.makedirs(MODELS_SAVE_PATH, exist_ok=True)
print(f"Directory '{MODELS_SAVE_PATH}' ensured to exist.")

# --- Save the two trained meta-models ---
lower_model_path = os.path.join(MODELS_SAVE_PATH, 'meta_model_xg_final_lower.joblib')
upper_model_path = os.path.join(MODELS_SAVE_PATH, 'meta_model_xg_final_upper.joblib')

joblib.dump(model_meta_final_lower, lower_model_path)
joblib.dump(model_meta_final_upper, upper_model_path)

print(f"\nMeta-models saved successfully!")
print(f"Lower bound model saved to: {lower_model_path}")
print(f"Upper bound model saved to: {upper_model_path}")

Directory './final_models/' ensured to exist.

Meta-models saved successfully!
Lower bound model saved to: ./final_models/meta_model_xg_final_lower.joblib
Upper bound model saved to: ./final_models/meta_model_xg_final_upper.joblib


In [12]:
# --- Create Final Submission File ---
print("\n--- Creating final submission file ---")
final_test_lower = model_meta_final_lower.predict(X_meta_test_enhanced)
final_test_upper = model_meta_final_upper.predict(X_meta_test_enhanced)

submission_df = pd.DataFrame({
    'id': pd.read_csv('./test.csv')['id'],
    'pi_lower': final_test_lower,
    'pi_upper': final_test_upper
})
submission_df['pi_lower'] = submission_df['pi_lower'].clip(0, None)
submission_df['pi_upper'] = np.maximum(submission_df['pi_lower'], submission_df['pi_upper'])

submission_filename = f'submission_enhanced_meta_{int(best_score_final)}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
display(submission_df.head())


--- Creating final submission file ---

'submission_enhanced_meta_290912.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,834994.43750,1.082122e+06
1,200001,564620.68750,7.761273e+05
2,200002,457883.15625,6.352390e+05
3,200003,284900.37500,4.154975e+05
4,200004,358065.28125,7.275292e+05
